In [1]:
"""MEMBERSHIP FUNCTIONS"""

def trimf(x,a,b,c):
    """Triangular membership function."""
    if x <= a or x >= c : return 0.0
    if a < x <= b: return (x - a) / (b - a)
    if b < x < c: return (c - x) / (c - b)
    return 0.0



def trapmf(x,a,b,c,d):
    """Trapezoidal membership function."""
    if x <= a or x >= d: return 0.0
    if a < x <= b: return (x - a) / (b - a) if b != a else 1.0
    if b < x <= c: return 1.0
    if c < x < d: return (d - x) / (d - c) if d != c else 1.0
    return 0.0

In [2]:

def fuzzify_stock(val):
    return {
        'low': trapmf(val, 0, 0, 15, 30),
        'medium': trimf(val, 15, 50, 85),
        'high': trapmf(val, 70, 85, 100, 100)
    }

def fuzzify_sales(val):
    return {
        'low': trapmf(val, 0, 0, 10, 20),
        'medium': trimf(val, 10, 25, 40),
        'high': trapmf(val, 30, 40, 50, 50)
    }

In [3]:
def evaluate_rules(stock, sales):
    # Calculate firing strength for all 9 combinations using MIN (Mamdani AND)
    # Output: LOW urgency rules
    r_med_low = min(stock['medium'], sales['low'])
    r_high_low = min(stock['high'], sales['low'])
    r_high_med = min(stock['high'], sales['medium'])
    
    # Output: MEDIUM urgency rules
    r_low_low = min(stock['low'], sales['low'])
    r_med_med = min(stock['medium'], sales['medium'])
    r_high_high = min(stock['high'], sales['high'])
    
    # Output: CRITICAL urgency rules
    r_low_med = min(stock['low'], sales['medium'])
    r_low_high = min(stock['low'], sales['high'])
    r_med_high = min(stock['medium'], sales['high'])
    
    # Aggregate (Union) using MAX
    return {
        'low': max(r_med_low, r_high_low, r_high_med),
        'medium': max(r_low_low, r_med_med, r_high_high),
        'critical': max(r_low_med, r_low_high, r_med_high)
    }

In [4]:
def defuzzify_urgency(rule_strengths):
    numerator = 0.0
    denominator = 0.0
    
    # Discretize the output universe from 0 to 100
    for x in range(101):
        # Determine the base membership of x in the output shapes
        mu_low = trapmf(x, 0, 0, 20, 40)
        mu_med = trimf(x, 20, 50, 80)
        mu_crit = trapmf(x, 60, 80, 100, 100)
        
        # Clip the shapes based on the rule strengths (Mamdani implication)
        clipped_low = min(mu_low, rule_strengths['low'])
        clipped_med = min(mu_med, rule_strengths['medium'])
        clipped_crit = min(mu_crit, rule_strengths['critical'])
        
        # Aggregate the clipped shapes
        aggregated_mu = max(clipped_low, clipped_med, clipped_crit)
        
        # Accumulate for Centroid calculation
        numerator += x * aggregated_mu
        denominator += aggregated_mu
        
    return numerator / denominator if denominator != 0 else 0.0

In [5]:
# 1. Inputs
current_stock = 12
daily_sales = 40

# 2. Fuzzification
stock_fuzzy = fuzzify_stock(current_stock)
sales_fuzzy = fuzzify_sales(daily_sales)

# 3. Rule Evaluation
strengths = evaluate_rules(stock_fuzzy, sales_fuzzy)

# 4. Defuzzification
final_urgency = defuzzify_urgency(strengths)

print(f"Current Stock: {current_stock}")
print(f"Daily Sales: {daily_sales}")
print(f"Restock Urgency: {final_urgency:.2f}%")

Current Stock: 12
Daily Sales: 40
Restock Urgency: 84.19%
